# Week 11 Coding

**C1** designs filters by truncation of the ideal impulse response and checks them against a specification. **C2** compares two filters with the same magnitude response and two differentiators. **C3** (optional) computes the moving average with a recursion and measures its rounding errors. **C4** (optional) tapers the truncated response with the Hann window and compares the two designs.

**Predict first.** Before running any check cell, write down on paper what the output will be.

**How to work in this notebook.** Execute the cells in order with Shift+Enter. Each task has a cell with gaps marked `...`: fill them, run the cell, then execute the check cell below it. A passing check prints a confirmation; a failing one raises an `AssertionError`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

## C1: Design by Truncation and Its Check (P4, P5)

The ideal low-pass response with cutoff $f_c$, that is $\Omega_c = 2\pi f_c/f_s$, is $h_d[n] = \sin(\Omega_c n)/(\pi n)$ for $n \neq 0$ and $h_d[0] = \Omega_c/\pi$. Truncation keeps $\lvert n\rvert \le M$, so that $N = 2M + 1$; the delay by $M$ makes the filter causal: $h[k] = h_d[k - M]$, $k = 0, \dots, N - 1$. The ideal high-pass response is $\delta[n] - h_d[n]$.

**a)** Complete `ideal_lowpass`, which returns $h_d[n]$ for $n = -M, \dots, M$, and `highpass`, which returns the causal coefficients of the high-pass filter of length $N$.

In [ ]:
def ideal_lowpass(fc, fs, M):
    """h_d[n] for n = -M, ..., M (centred)."""
    n = np.arange(-M, M + 1)
    Wc = 2 * np.pi * fc / fs                                                        # your code: the cutoff in rad/sample
    hd = np.zeros(2 * M + 1)
    nz = n != 0
    hd[nz] = np.sin(np.pi/2 * n[nz]) / (n[nz] * np.pi)                                                    # your code: sin(Wc n)/(pi n) for n != 0
    hd[~nz] = Wc / np.pi                                                   # your code: the value at n = 0
    return hd


def highpass(N, fc, fs):
    """Causal high-pass filter of odd length N by truncation: delta[k - M] - h_d[k - M]."""
    M = (N - 1) // 2
    impulse = np.zeros(N)
    impulse[M] = 1.0
    return impulse - ideal_lowpass(fc, fs, M)                                                      # your code

In [ ]:
hd3 = ideal_lowpass(1, 4, 3)                                       # cutoff fs/4, that is Omega_c = pi/2 (P4)
assert np.allclose(hd3, [-1 / (3 * np.pi), 0, 1 / np.pi, 0.5, 1 / np.pi, 0, -1 / (3 * np.pi)]), "P4 a): h_d for |n| <= 3"
hp = highpass(29, 1250, 16000)
assert len(hp) == 29 and np.isclose(hp[14], 1 - 0.15625) and np.allclose(hp, hp[::-1]), "P5: 29 symmetric coefficients"
print("a) passed: h_d of P4 and the high-pass filter of P5.")

**b) The truncated filters of P4.** Before running the cell, write down your values of $H(0)$ and $H(\pi)$ for $N = 3$ and $N = 7$. The cell computes $H(0) = \sum_n h[n]$ and $H(\pi) = \sum_n (-1)^n h[n]$ for every odd $N$ up to $41$ and plots $H(0)$ against $N$.

In [ ]:
Ns = np.arange(3, 42, 2)
H0, Hpi = [], []
for N in Ns:
    M = (N - 1) // 2
    hd = ideal_lowpass(1, 4, M)
    sign = (-1.0) ** np.arange(-M, M + 1)
    H0.append(...)                                                  # your code: H(0)
    Hpi.append(...)                                                 # your code: H(pi)
H0, Hpi = np.array(H0), np.array(Hpi)
for N, a, b in zip(Ns[:6], H0, Hpi):
    print(f"N = {N:2d}: H(0) = {a:.3f}, H(pi) = {b:.3f}")
plt.figure(figsize=(7, 2.8))
plt.plot(Ns, H0, "o-", label="H(0)"); plt.axhline(1, color="0.6", lw=0.8)
plt.xlabel("N"); plt.legend(); plt.tight_layout()

In [ ]:
assert abs(H0[0] - 1.137) < 1e-3 and abs(H0[2] - 0.924) < 1e-3 and abs(Hpi[2] - 0.076) < 1e-3, "P4 b)"
assert np.isclose(H0[1], H0[0]), "P4 d): N = 5 adds two zero coefficients"
assert np.allclose(H0 + Hpi, 1), "H(0) + H(pi) = 2 h_d[0] = 1 for every N"
print("b) passed: the values of P4; H(0) oscillates about 1 and approaches it slowly.")

**c) The high-pass filter of P5.** The specification of the field recorder: passband above $1.5$ kHz with $0.85 \le \lvert H\rvert \le 1.15$, stopband below $1$ kHz with $\lvert H\rvert \le 0.15$, at $f_s = 16$ kHz. The function `check` (given) evaluates $\lvert H\rvert$ on a fine grid and returns the smallest and the largest value in the passband and the largest value in the stopband. Check the filter with $N = 29$, then find the shortest odd filter length that meets the specification.

In [ ]:
fs5 = 16000
f5 = np.linspace(0, fs5 / 2, 32001)


def check(h, fs=fs5, f=f5):
    """(smallest |H| in the passband, largest |H| in the passband, largest |H| in the stopband)."""
    A = np.abs(signal.freqz(h, worN=f, fs=fs)[1])
    return A[f >= 1500].min(), A[f >= 1500].max(), A[f <= 1000].max()


def meets(h):
    lo, hi, st = check(h)
    return ...                                                      # your code: True if all three bounds hold


print("N = 29:", np.round(check(highpass(29, 1250, fs5)), 3), meets(highpass(29, 1250, fs5)))
N_min = ...                                                         # your code: the smallest odd N >= 3 that meets the specification
print("shortest odd length:", N_min, np.round(check(highpass(N_min, 1250, fs5)), 3))

In [ ]:
lo, hi, st = check(highpass(29, 1250, fs5))
assert abs(lo - 0.884) < 2e-3 and abs(hi - 1.092) < 2e-3 and abs(st - 0.120) < 2e-3, "P5 c): N = 29"
assert N_min == 23, "the shortest odd length is 23"
print("c) passed: N = 29 meets the specification; so does every odd length from 23 on.")

**d) The longer filter of P5 d).** Compute the largest $\lvert H\rvert$ of the filter with $N = 201$ and the frequency at which it occurs, and plot both magnitude responses with the bounds of the specification.

In [ ]:
h201 = highpass(201, 1250, fs5)
A201 = np.abs(signal.freqz(h201, worN=f5, fs=fs5)[1])
peak, f_peak = ..., ...                                             # your code: the largest |H| and its frequency
print(f"N = 201: largest |H| = {peak:.3f} at {f_peak:.0f} Hz;", np.round(check(h201), 3))

plt.figure(figsize=(8, 3))
for N in (29, 201):
    plt.plot(f5, np.abs(signal.freqz(highpass(N, 1250, fs5), worN=f5, fs=fs5)[1]), label=f"N = {N}")
plt.fill_between([1500, 8000], 0.85, 1.15, color="green", alpha=0.12)
plt.fill_between([0, 1000], 0, 0.15, color="red", alpha=0.12)
plt.xlim(0, 4000); plt.xlabel("frequency [Hz]"); plt.ylabel("|H|"); plt.legend(); plt.tight_layout()

In [ ]:
assert abs(peak - 1.091) < 2e-3 and 1000 < f_peak < 1500, "the overshoot of N = 201 lies in the transition band"
print("d) passed: the largest |H| of N = 201 lies between 1 and 1.5 kHz.")

**Questions.**
1. Why does the estimate $N \approx 0.9 f_s/\Delta f = 29$ exceed the shortest length that meets this specification?
2. In the plot, where does the overshoot of the filter with $N = 29$ lie, and where that of the filter with $N = 201$? Which of the two is closer to the cutoff?

## C2: Phase and the Shape of a Signal (P6, P7)

**a) Two filters with the same magnitude response (P6).** The filters $h_1 = \{2, 5, 2\}/9$ and $h_2 = \{4, 4, 1\}/9$ have the same $\lvert H\rvert$. A symmetric signal, centred at $n = 15$, passes through both. Compute the two outputs, and the centre $\sum_n n\,y[n] / \sum_n y[n]$ of each output. Before running the cell, predict which output stays symmetric and where its centre lies.

In [ ]:
h1, h2 = np.array([2, 5, 2]) / 9, np.array([4, 4, 1]) / 9
n = np.arange(40)
u = np.exp(-0.5 * ((n - 15) / 3) ** 2)                              # a symmetric signal centred at n = 15
y1 = ...                                                            # your code: the output of h1, first 40 samples
y2 = ...                                                            # your code: the output of h2, first 40 samples
centre = lambda y: (np.arange(len(y)) * y).sum() / y.sum()
print(f"centres: input {centre(u):.2f}, h1 {centre(y1):.2f}, h2 {centre(y2):.2f}")
plt.figure(figsize=(8, 2.8))
plt.plot(n, u, "k:", label="input"); plt.plot(n, y1, label="h1"); plt.plot(n, y2, label="h2")
plt.xlim(0, 32); plt.xlabel("n"); plt.legend(); plt.tight_layout()

In [ ]:
W = np.linspace(0, np.pi, 501)
assert np.allclose(np.abs(signal.freqz(h1, worN=W)[1]), np.abs(signal.freqz(h2, worN=W)[1])), "the same |H|"
assert np.allclose(y1[6:27], y1[6:27][::-1]), "y1 is symmetric about n = 16"
assert abs(centre(y1) - 16) < 1e-3 and abs(centre(y2) - 15.667) < 1e-3, "centres 16 and 15.67"
assert not np.allclose(y2[6:27], y2[6:27][::-1]), "y2 is not symmetric"
print("a) passed: h1 delays the signal by one sample; h2 moves it by 0.67 samples and makes it asymmetric.")

**b) Two differentiators (P7).** At $f_s = 100$ Hz, the central difference $y[n] = (u[n] - u[n-2])/(2T)$ and the backward difference $y[n] = (u[n] - u[n-1])/T$ estimate the velocity from the angle $u[n]$. Complete `ratio`, the magnitude of a filter divided by the magnitude $\omega = 2\pi f$ of the exact derivative, and evaluate it at several frequencies. Then add measurement noise with a standard deviation of $0.001$ rad to a constant angle and compare the standard deviations of the two velocity estimates.

In [ ]:
fs7 = 100
T = 1 / fs7
central = np.array([1, 0, -1]) / (2 * T)
backward = np.array([1, -1]) / T


def ratio(h, f):
    """|H| at f in hertz divided by the magnitude 2 pi f of the exact derivative."""
    return ...                                                      # your code


for f in (1, 3.9, 10, 25):
    print(f"{f:4} Hz: central {ratio(central, f):.3f}, backward {ratio(backward, f):.3f}")

rng = np.random.default_rng(0)
noise = 0.001 * rng.normal(size=200000)                             # a constant angle with measurement noise
std_central = np.std(np.convolve(noise, central, mode="valid"))
std_backward = ...                                                  # your code: the same for the backward difference
print(f"noise of the velocity: central {std_central:.4f} rad/s, backward {std_backward:.4f} rad/s")

In [ ]:
assert abs(ratio(central, 3.9) - 0.990) < 1e-3 and abs(ratio(central, 25) - 2 / np.pi) < 1e-6, "P7 b): sin(Omega)/Omega"
assert abs(std_central - 0.0707) < 2e-3 and abs(std_backward - 0.1414) < 3e-3, "the noise of the two estimates"
print("b) passed: the central difference is accurate to 1 % up to 3.9 Hz and halves the noise of the backward difference.")

**Questions.**
1. Why does the backward difference follow the exact derivative more closely at $25$ Hz, and yet give the noisier estimate?
2. The standard deviation of the noise at the output of an FIR filter is that of the input times $\sqrt{\sum_k h[k]^2}$. Verify the two values with this rule.

## C3 (optional): The Moving Average Computed by a Recursion (P11)

The recursion $y[n] = y[n-1] + (u[n] - u[n-N])/N$ computes the moving average with two additions and one division per sample, whatever $N$. A barometer delivers air pressure close to $1000$ hPa with small fluctuations. Complete the recursion in single precision (`np.float32`, as on many sensors), and compare it with the direct sum over $N$ samples, computed in double precision.

In [ ]:
N = 100
L = 1_000_000
rng = np.random.default_rng(1)
x = (1000 + rng.normal(size=L)).astype(np.float32)                  # air pressure in hPa
xp = np.concatenate([np.zeros(N, np.float32), x])                   # u[n] = 0 for n < 0
d = (xp[N:] - xp[:-N]) / np.float32(N)                              # (u[n] - u[n - N])/N, single precision

y = np.empty(L, np.float32)
s = np.float32(0)
for i in range(L):                                                  # about one second
    s = ...                                                         # your code: the recursion, in single precision
    y[i] = s

direct = np.convolve(x.astype(np.float64), np.ones(N) / N)[:L]      # the direct sum, double precision
err = np.abs(y - direct)
print(f"largest error: first 10^4 samples {err[:10000].max():.1e} hPa, last 10^4 samples {err[-10000:].max():.1e} hPa")

In [ ]:
direct32 = np.convolve(x, np.ones(N, np.float32) / np.float32(N))[:L]           # the direct sum, single precision
err32 = np.abs(direct32 - direct)
assert err[-10000:].max() > 5 * err[:10000].max(), "the error of the recursion grows"
assert err32[-10000:].max() < 5 * err32[:10000].max() + 1e-4, "the error of the direct sum does not grow"
print(f"passed: the error of the recursion grows; the direct sum in single precision stays at {err32.max():.1e} hPa.")

**Questions.**
1. Why does the error of the recursion grow along the signal, while that of the direct sum does not?
2. Two remedies: compute the recursion in double precision, or recompute the sum directly every $10\,000$ samples. Why does each of them work?

## C4 (optional): Tapering the Truncated Response (P5, P8)

Truncation cuts the ideal response off abruptly, and its overshoot of about $9\,\%$ remains for every $N$ (C1 d). As for the spectrogram in Week 9, the abrupt ends can be replaced by a smooth taper: the truncated coefficients are multiplied by the symmetric Hann window `np.hanning(N)`, $w[k] = \tfrac12\bigl(1 - \cos(2\pi k/(N-1))\bigr)$ for $k = 0, \dots, N-1$, which falls to zero at both ends.

The next cell defines the truncated high-pass filter of P5 and the function `check` of C1 c). Run it first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs5 = 16000                                                         # the field recorder of P5
f5 = np.linspace(0, fs5 / 2, 32001)


def truncated(N, fc=1250, fs=fs5):
    """The high-pass filter of P5 by truncation (C1), odd length N: delta[n] - sin(Wc n)/(pi n), delayed by (N - 1)/2."""
    n = np.arange(N) - (N - 1) // 2
    return (n == 0) * 1.0 - 2 * fc / fs * np.sinc(2 * fc / fs * n)   # np.sinc(x) = sin(pi x)/(pi x)


def check(h, fs=fs5, f=f5):
    """(smallest |H| in the passband, largest |H| in the passband, largest |H| in the stopband), as in C1 c)."""
    A = np.abs(signal.freqz(h, worN=f, fs=fs)[1])
    return A[f >= 1500].min(), A[f >= 1500].max(), A[f <= 1000].max()

**a)** Complete `tapered`, choose an odd filter length `N`, for example $201$, and compare the tapered and the truncated filter. Before running the cell, predict what the taper does to the overshoot and to the transition band. Then try $N = 29$ and $N = 61$.

In [ ]:
def tapered(N):
    """The truncated filter of length N, multiplied by the Hann window."""
    return ...                                                      # your code


N = ...                                                             # your choice: an odd length, e.g. 201; try also 29 and 61
h_trunc, h_hann = truncated(N), tapered(N)
print("truncated:", np.round(check(h_trunc), 4), "  Hann:", np.round(check(h_hann), 4))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for h, label in ((h_trunc, "truncated"), (h_hann, "Hann")):
    A = np.abs(signal.freqz(h, worN=f5, fs=fs5)[1])
    ax[0].plot(f5, A, label=label)
    ax[1].plot(f5, 20 * np.log10(A), label=label)
ax[0].fill_between([1500, 8000], 0.85, 1.15, color="green", alpha=0.12)
ax[0].fill_between([0, 1000], 0, 0.15, color="red", alpha=0.12)
ax[0].set(xlim=(0, 3000), xlabel="frequency [Hz]", ylabel="|H|")
ax[1].set(xlim=(0, 3000), ylim=(-80, 5), xlabel="frequency [Hz]", ylabel="|H| in dB")
ax[0].legend(); plt.tight_layout()

In [ ]:
hw = tapered(N)
assert len(hw) == N and N % 2 == 1 and np.allclose(hw, hw[::-1]), "an odd number N of symmetric coefficients"
assert hw[0] == 0 and np.isclose(hw[(N - 1) // 2], 1 - 0.15625), "zero at both ends, centre coefficient unchanged"
A_hann = np.abs(signal.freqz(hw, worN=f5, fs=fs5)[1])
assert N >= 11 and abs(A_hann.max() - 1.006) < 1e-3, "an overshoot of about 0.6 % (choose N >= 11)"
print(f"a) passed for N = {N}: with the taper, the largest |H| is {A_hann.max():.3f}; by truncation, it is about 1.09.")

**b) Which design is shorter?** Find the shortest odd length of both designs for the specification of P5 (tolerance $0.15$), then for a tighter one with the same band edges: $0.99 \le \lvert H\rvert \le 1.01$ in the passband and $\lvert H\rvert \le 0.01$ in the stopband (tolerance $0.01$). Before running the cell, predict which design is shorter in each case.

In [ ]:
def meets_tol(h, tol):
    """True if the passband lies between 1 - tol and 1 + tol, and the stopband below tol."""
    lo, hi, st = check(h)
    return lo >= 1 - tol and hi <= 1 + tol and st <= tol


lengths = {}
for tol in (0.15, 0.01):                                            # a few seconds
    N_trunc = next(N for N in range(3, 1001, 2) if meets_tol(truncated(N), tol))
    N_hann = ...                                                    # your code: the same for tapered
    lengths[tol] = N_trunc, N_hann
    print(f"tolerance {tol}: truncated N = {N_trunc}, Hann N = {N_hann}")

In [ ]:
assert lengths[0.15] == (23, 53) and lengths[0.01] == (649, 99), "the shortest odd lengths"
print("b) passed: for the tolerance of 15 % (P5), truncation needs 23 taps and the taper 53; for the tolerance of 1 %, truncation needs 649 and the taper 99.")

**Questions.**
1. Why does the taper lower the overshoot, and what does it cost? Compare with the Hann window of the spectrogram (Week 9) and with the two moving average filters in series of P8.
2. With `N` set to $29$, $61$ and $201$ in a): does the overshoot of either design depend on $N$? Use the answer to explain the lengths found in b).
3. Why does the tapered filter still have linear phase, with the same delay?